# HPT Prototype — Colab runner

Runs the Healthcare Price Transparency pipeline from the modularized repo.
All Colab-specific glue (pip install, Drive mount, `cp` + `tar -xzf`) lives in
`colab_run.py` at the repo root; this notebook is a thin driver.

**Runtime → Change runtime type → GPU** (MedGemma 1.5-4B needs it).

You must already have these in your Drive (same locations as the prototype):
- `MyDrive/New_PT_DB_Backups/New_PT_DB.tar.gz` — Qdrant snapshot
- `MyDrive/Price_Transparency/datasets/everyUPMCmrf_clean (1).csv` — cleaned MRF

and a `HUGGINGFACE_API_KEY` (a `.env`, or a Colab secret of the same name).

## 1. Get the code and dependencies

Equivalent to the original notebook's `!pip install ...` cell.

In [ ]:
REPO = "/content/HPT_Prototype"

import os
if not os.path.isdir(REPO):
    !git clone https://github.com/adi274903/HPT_Prototype "$REPO"
%cd "$REPO"

# The prototype's install line, plus pandas / python-dotenv:
!pip install -q -r requirements.txt

## 2. Mount Drive and stage the Qdrant snapshot

This is the `drive.mount('/content/drive')` + `!cp` + `!tar -xzf` part of the
original notebook.

In [ ]:
import colab_run

colab_run.mount_drive()                 # drive.mount('/content/drive')
qdrant_path = colab_run.stage_db()      # cp + tar -xzf -> /content/New_PT_DB

print(colab_run.default_paths())

## 3. Load the models

MedGemma (image-text-to-text, bfloat16 on CUDA) + the MedTE sentence embedder.
Takes a few minutes the first time as the weights download.

In [ ]:
pipeline = colab_run.setup(mount=False, stage=False, qdrant_path=qdrant_path)

## 4. Ask a question

In [ ]:
user_query = (
    "What might a diagnostic mammogram cost at UPMC Presbyterian "
    "with UPMC Health Plan?"
)

result = pipeline.run(user_query)
print(result["answer"])

## 5. The notebook's other validation queries

In [ ]:
user_query = "Best and cheapest hospital for me to get my colonoscopy done with Highmark BCBS? "
pipeline.run(user_query)

In [ ]:
user_query = "Best and cheapest hospital for me to get my mammogram done? "
pipeline.run(user_query)

## Inspecting a result

`pipeline.run(...)` returns the full trace, so any step can be inspected:

In [ ]:
result.keys()
# result['categorized']       # medical / hospital / insurer / medication entities
# result['cpt_candidates']    # retrieved CPT candidates (rank, score, code, text)
# result['hcpcs_candidates']  # retrieved HCPCS candidates
# result['output']            # validated selected codes (post code-check)
# result['code_plausible']    # MRF matches + price_summary + filters_used
# result['answer']            # final patient-facing text
# result['total_seconds']

## One-liner alternative

If you would rather not run cells 2–3 separately:

In [ ]:
# import colab_run
# result = colab_run.run("What might a diagnostic mammogram cost at UPMC Presbyterian?")
#
# ...which mounts Drive, stages the DB, loads the models and answers in one call.
# From a shell cell instead:  !python colab_run.py